# Bài 6 · Kết nối & truy xuất dữ liệu ngoài

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Đọc file "khó" bằng `read_csv` (nén gz, chọn cột, ép kiểu, parse ngày) và dùng **Parquet**
   cho dữ liệu trung gian.
2. Gọi một API thật bằng `requests` (params, timeout, status), biến JSON thành DataFrame,
   và **lưu response thô** để tái lập.
3. Viết truy vấn SQL cơ bản (SELECT/WHERE/GROUP BY/JOIN) chạy bằng **DuckDB** —
   trực tiếp trên file CSV/Parquet.

In [ ]:
%pip install -q duckdb

import json, time
from pathlib import Path
import pandas as pd
import requests
import duckdb

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

## 1. Đọc file đúng cách

Lần này ta đọc bảng `listings` **đầy đủ** (90 cột, nén gz) của Santiago — nhưng chỉ lấy đúng
những cột cần. Chú ý: file nén `.gz` đọc thẳng, không phải giải nén.

In [ ]:
URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")

df = pd.read_csv(
    URL_FULL,
    usecols=["id", "neighbourhood_cleansed", "room_type", "price",
             "host_since", "minimum_nights", "review_scores_rating"],
    parse_dates=["host_since"],
)
print(df.shape)
df.dtypes

`price` vẫn là **chuỗi** — bảng đầy đủ lưu giá dạng `"$45,647.00"` (khác bản rút gọn tuần trước).
pandas đoán kiểu giỏi, nhưng không đoán hộ được quy ước tiền tệ. Quen mặt ai chưa? `clean_price`!

In [ ]:
df["price_num"] = (df["price"]
                   .str.replace("$", "", regex=False)
                   .str.replace(",", "", regex=False)
                   .astype(float))
df[["price", "price_num"]].head(3)

### CSV vs Parquet — đo trên dữ liệu thật

In [ ]:
import os, time

df.to_csv("data/processed/listings.csv", index=False)
df.to_parquet("data/processed/listings.parquet")

for f in ["listings.csv", "listings.parquet"]:
    path = f"data/processed/{f}"
    t0 = time.perf_counter()
    _ = pd.read_parquet(path) if f.endswith("parquet") else pd.read_csv(path)
    t = (time.perf_counter() - t0) * 1000
    print(f"{f:22} {os.path.getsize(path)/1e6:5.1f} MB   đọc lại {t:5.0f} ms")

In [ ]:
# Parquet còn giữ dtype: host_since đọc lại vẫn là datetime, không phải chuỗi
pd.read_parquet("data/processed/listings.parquet").dtypes[["host_since", "price_num"]]

## 2. Gọi API thật: thời tiết Open-Meteo

[Open-Meteo](https://open-meteo.com) là API thời tiết miễn phí, **không cần key** — sân tập
hoàn hảo. Lấy dự báo nhiệt độ Santiago 7 ngày tới:

In [ ]:
r = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": -33.45, "longitude": -70.66,     # Santiago
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "auto",
    },
    timeout=20,          # LUÔN có timeout
)
r.raise_for_status()      # lỗi thì dừng ngay tại đây
print("Status:", r.status_code)

In [ ]:
d = r.json()              # dict lồng nhau
print(list(d.keys()))
print(json.dumps(d["daily"], ensure_ascii=False)[:150], "…")

In [ ]:
# Lưu response THÔ vào raw/ kèm ngày giờ — bằng chứng để tái lập
tem = pd.Timestamp.now().strftime("%Y%m%d")
raw_path = Path(f"data/raw/openmeteo_santiago_{tem}.json")
raw_path.write_text(json.dumps(d, ensure_ascii=False))
print("Đã lưu:", raw_path)

# Từ đây trở đi: làm việc với file đã lưu, không gọi mạng lại
d2 = json.loads(raw_path.read_text())
thoi_tiet = pd.DataFrame(d2["daily"])
thoi_tiet

Ba "phép lịch sự" trong cell trên mà bạn sẽ mang theo cả kỳ: `timeout=20`,
`raise_for_status()`, và **lưu raw trước khi xử lý**. Khi gọi API trong vòng lặp, thêm
`time.sleep(1)` giữa các request.

## 3. SQL & DuckDB

DuckDB là cơ sở dữ liệu phân tích chạy **ngay trong notebook** — không cài server, và
query được **thẳng file CSV/Parquet**.

In [ ]:
duckdb.query("""
    SELECT room_type,
           median(price_num) AS gia_trung_vi,
           count(*)          AS n
    FROM 'data/processed/listings.parquet'
    GROUP BY room_type
    ORDER BY gia_trung_vi DESC
""").df()

In [ ]:
# KIỂM CHỨNG CHÉO: pandas phải ra đúng số đó
df.groupby("room_type")["price_num"].agg(["median", "size"])

Hai công cụ độc lập, một đáp số — đây là cách kiểm chứng rẻ nhất (và là chiêu tốt để kiểm tra
code AI viết). Giờ thử JOIN với một DataFrame đang sống trong notebook:

In [ ]:
vung_df = pd.DataFrame({
    "neighbourhood_cleansed": ["Santiago", "Providencia", "Las Condes", "Ñuñoa"],
    "vung": ["Trung tâm", "Đông", "Đông", "Đông"],
})

duckdb.query("""
    SELECT v.vung,
           median(l.price_num) AS gia,
           count(*)            AS n
    FROM 'data/processed/listings.parquet' AS l
    LEFT JOIN vung_df AS v USING (neighbourhood_cleansed)
    GROUP BY v.vung
    ORDER BY gia
""").df()

DuckDB "nhìn thấy" `vung_df` của Python — trộn SQL với pandas tuỳ thích. Dòng `NULL`/`NaN`
là các quận chưa có trong bảng tra cứu (hệt left-merge tuần trước).

## 4. Bài tập tại lớp

### Bài 1 — read_csv có chủ đích

Bảng `reviews` **rút gọn** của Santiago chỉ có 2 cột (`listing_id`, `date`):
`https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv`

Đọc nó với `parse_dates=["date"]`, rồi đếm số review theo **năm**
(gợi ý: `df["date"].dt.year` — buổi 8 học kỹ, giờ cứ dùng).
Năm nào nhiều review nhất?

In [ ]:
# TODO Bài 1:
rv = pd.read_csv(
    "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/reviews.csv",
    parse_dates=["date"],
)
rv["date"].dt.year.value_counts().sort_index().tail(5)

### Bài 2 — Gọi API cho thành phố khác

Rio de Janeiro nằm ở (`-22.91`, `-43.17`). Gọi Open-Meteo lấy nhiệt độ max 7 ngày tới của Rio,
rồi ghép với Santiago thành một bảng hai cột để so sánh (gợi ý: hai DataFrame,
`merge` theo `time`). Đông ở Nam Mỹ — thành phố nào "ấm" hơn tuần này?

In [ ]:
# TODO Bài 2:
r2 = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": -22.91, "longitude": -43.17,
            "daily": "temperature_2m_max", "timezone": "auto"},
    timeout=20,
)
r2.raise_for_status()
rio = pd.DataFrame(r2.json()["daily"]).rename(columns={"temperature_2m_max": "rio_max"})
scl = thoi_tiet[["time", "temperature_2m_max"]].rename(columns={"temperature_2m_max": "santiago_max"})
so_sanh = scl.merge(rio, on="time")
so_sanh

### Bài 3 — SQL tự viết

Viết **một** truy vấn DuckDB trên `listings.parquet` trả về: các quận
(`neighbourhood_cleansed`) có **ít nhất 500 listing**, kèm giá trung vị và số listing,
xếp giảm dần theo giá trung vị. (Gợi ý: `HAVING count(*) >= 500`.)
Đối chiếu kết quả với chuỗi pandas tương đương ở buổi 5.

In [ ]:
# TODO Bài 3:
duckdb.query("""
    SELECT neighbourhood_cleansed,
           median(price_num) AS gia_trung_vi,
           count(*)          AS n
    FROM 'data/processed/listings.parquet'
    GROUP BY neighbourhood_cleansed
    HAVING count(*) >= 500
    ORDER BY gia_trung_vi DESC
""").df().head(5)

## 5. Thử thách về nhà 🏆 — Script thu thập kiểu bài tập lớn

Viết script (hàm) `download_snapshot(city_path, date, dest)`:

1. Tải 3 file `listings.csv.gz`, `reviews.csv.gz`, `neighbourhoods.geojson` của một snapshot
   Inside Airbnb về `data/raw/<city>/<date>/` (dùng `requests`, ghi bytes; **bỏ qua nếu file
   đã tồn tại** — tải một lần thôi).
2. Kiểm tra tối thiểu: file tồn tại, đọc được, số dòng > 0 — in báo cáo ngắn.
3. Chạy thử với thành phố nhóm bạn định chọn cho bài tập lớn.

Đây chính là bước "Thu thập" trong đề — làm xong từ bây giờ, sau này đỡ một việc.

In [ ]:
RUN_CHALLENGE = False

if RUN_CHALLENGE:
    def download_snapshot(city_path, date, dest="data/raw"):
        base = f"https://data.insideairbnb.com/{city_path}/{date}"
        ...
    download_snapshot("chile/rm/santiago", "2026-06-29")

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Giữ nguyên dữ liệu raw; xử lý ghi ra `processed/` | Pipeline chạy lại được — cơ chế chấm bài tập lớn |
| `usecols`/`parse_dates`/`na_values`; gz đọc thẳng; Parquet giữ dtype | Đọc nhanh, đúng kiểu, ít RAM |
| API: params + timeout + raise_for_status + sleep + **lưu raw** | Không treo, không bị chặn, tái lập được |
| DuckDB query thẳng CSV/Parquet, JOIN cả DataFrame | SQL không cần server; kiểm chứng chéo với pandas |

**Buổi sau:** dữ liệu chuỗi ký tự — `str.` và regex, vũ khí chính khi dữ liệu là văn bản.